# Gradient Diagnostics: Forward-Backward Gain Asymmetry

This notebook investigates **why** row-centered initializations cause gradient instability
despite preserving geometry. The analysis is motivated by the 4 backpropagation equations
and the advisor's observations:

1. **Single data point test**: Does the gradient pattern hold for a single sample? If yes,
   the issue is in σ'(z^l) and the weight structure, not aggregation.

2. **Forward-backward gain decomposition**: Since ∂C/∂W^l = δ^l · (a^{l-1})^T,
   we decompose the gradient into its forward (activation) and backward (error signal)
   components to identify the source of instability.

3. **Centering ratio**: Var(a)/E[a²] measures how much row-centering reduces the
   effective forward gain. For half-Gaussian post-ReLU: ≈ (π-1)/π ≈ 0.68.

4. **Variance sweep**: Confirms that no single variance fixes both forward and backward
   gains (the advisor's coupling observation from BP4).

5. **Alpha sweep**: Explores the geometry-gradient Pareto frontier for partial centering.

In [ ]:
# Setup
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
import torch
import math

from rp_study.config import ExperimentConfig, NetworkConfig, GradientExperimentConfig
from rp_study.experiments.gradient_analysis import (
    GradientExperiment, compare_initializations, ExperimentResults
)
from rp_study.visualization.gradient_plots import (
    plot_gain_decomposition, plot_signal_norms, compare_gains_plot,
    plot_relu_survival, plot_centering_ratio, compare_initializations_plot,
    plot_row_norm_per_layer,
)
from rp_study.models.initializers import list_initializers

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"Available initializers: {list_initializers()}")

In [ ]:
# ====== CONFIGURATION ======

SEED = 42
DATASET = "fashion_mnist"
DATA_DIR = "../data"

# Initializers to compare
INIT_STRATEGIES = [
    "he",                              # Baseline: good gradients, bad geometry
    "row_centered_he_var_adj",          # Good geometry, bad gradients
    "row_centered_forward_balanced",    # Diagnostic: forward gain = 1
]

INIT_LABELS = {
    "he": "He (baseline)",
    "row_centered_he_var_adj": "Row-Centered (var adj)",
    "row_centered_forward_balanced": "Row-Centered (fwd balanced)",
    "partial_centered_he": "Partial Centered",
}

# Network architecture
N_HIDDEN = 50
WIDTH = 784
LAYER_SIZES = [784] + [WIDTH] * N_HIDDEN + [1]

# Sample counts
NUM_SAMPLES_FULL = 2000
NUM_SAMPLES_SINGLE = 1

print(f"Architecture: {LAYER_SIZES[0]} -> [{WIDTH}] x {N_HIDDEN} -> {LAYER_SIZES[-1]}")
print(f"Initializers: {INIT_STRATEGIES}")

## Section 1: Single vs Multi-Sample Comparison

**Advisor's suggestion (BP2)**: If the gradient pattern (explosion/vanishing) holds for
a single data point, then the cause is structural — in σ'(z^l) and the weight matrices —
not in the aggregation over many samples.

We compare gradient row norms with `num_samples=1` vs `num_samples=2000`.

In [ ]:
# Run with full batch
print(f"Running gradient analysis with {NUM_SAMPLES_FULL} samples...")
results_full = compare_initializations(
    layer_sizes=LAYER_SIZES,
    init_strategies=INIT_STRATEGIES,
    num_samples=NUM_SAMPLES_FULL,
    dataset=DATASET,
    seed=SEED,
)
print("Done!")

In [ ]:
# Run with single data point
print(f"Running gradient analysis with {NUM_SAMPLES_SINGLE} sample...")
results_single = compare_initializations(
    layer_sizes=LAYER_SIZES,
    init_strategies=INIT_STRATEGIES,
    num_samples=NUM_SAMPLES_SINGLE,
    dataset=DATASET,
    seed=SEED,
)
print("Done!")

In [ ]:
# Compare gradient norm profiles: single vs multi-sample
for init in INIT_STRATEGIES:
    label = INIT_LABELS.get(init, init)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

    # Full batch
    norms_full = results_full[init].get_mean_row_norms()
    layers = list(norms_full.keys())[:-1]  # exclude output layer
    vals_full = [norms_full[k] for k in layers]
    ax1.plot(layers, vals_full, 'o-', markersize=3)
    ax1.set_yscale('log')
    ax1.set_title(f'{label}: {NUM_SAMPLES_FULL} samples')
    ax1.set_xlabel('Layer')
    ax1.set_ylabel('Mean Gradient Row Norm')
    ax1.grid(True, alpha=0.3)

    # Single sample
    norms_single = results_single[init].get_mean_row_norms()
    vals_single = [norms_single[k] for k in layers]
    ax2.plot(layers, vals_single, 's-', color='tab:orange', markersize=3)
    ax2.set_yscale('log')
    ax2.set_title(f'{label}: {NUM_SAMPLES_SINGLE} sample')
    ax2.set_xlabel('Layer')
    ax2.set_ylabel('Mean Gradient Row Norm')
    ax2.grid(True, alpha=0.3)

    # Thin x-axis for readability
    for ax in [ax1, ax2]:
        if len(layers) > 10:
            step = max(1, (len(layers) - 1) // 9)
            visible = list(range(0, len(layers), step))
            if len(layers) - 1 not in visible:
                visible.append(len(layers) - 1)
            ax.set_xticks([layers[i] for i in visible])

    plt.suptitle(f'{label}: Single vs Multi-Sample Gradient Profile', fontsize=13)
    plt.tight_layout()
    plt.show()

    # Print correlation
    if len(vals_full) > 0 and len(vals_single) > 0:
        log_corr = np.corrcoef(np.log(np.array(vals_full) + 1e-20),
                               np.log(np.array(vals_single) + 1e-20))[0, 1]
        print(f'  {label}: log-scale correlation = {log_corr:.4f}'
              f' ({"same pattern" if log_corr > 0.9 else "different pattern"})')

## Section 2: Forward-Backward Gain Decomposition

Since ∂C/∂W^l = δ^l · (a^{l-1})^T (BP4), the gradient norm is proportional to
||δ^l|| · ||a^{l-1}||. By tracking these separately, we can identify:

- **Forward signal**: rms(a^l) — does it grow, shrink, or stay constant with depth?
- **Backward signal**: rms(δ^l) — does the error signal explode or vanish?
- **Centering ratio**: Var(a)/E[a²] — the "smoking gun" for the forward-backward asymmetry

In [ ]:
# Signal norm decomposition for each initializer
for init in INIT_STRATEGIES:
    label = INIT_LABELS.get(init, init)
    result = results_full[init]

    print(f'\n=== {label} ===')
    fig = plot_signal_norms(result, use_log_scale=True)
    plt.suptitle(f'{label}: Signal Norm Decomposition', fontsize=13, y=1.02)
    plt.show()

    fig = plot_gain_decomposition(result)
    plt.suptitle(f'{label}: Per-Layer Gains', fontsize=13, y=1.02)
    plt.show()

In [ ]:
# Compare forward gains across all initializers
results_labeled = {INIT_LABELS.get(k, k): v for k, v in results_full.items()}

fig = compare_gains_plot(results_labeled, gain_type='forward')
plt.title('Forward Gain Comparison (should be ~1.0 for stability)', fontsize=12)
plt.show()

fig = compare_gains_plot(results_labeled, gain_type='backward')
plt.title('Backward Gain Comparison (should be ~1.0 for stability)', fontsize=12)
plt.show()

In [ ]:
# Centering ratio: the "smoking gun"
fig = plot_centering_ratio(results_labeled)
plt.title('Centering Ratio Var(a)/E[a²] — Forward Gain Reduction Factor', fontsize=12)
plt.show()

print('Interpretation:')
print('  - This ratio is a property of the ACTIVATIONS, not the weights')
print('  - For standard He: the ratio does NOT affect forward gain (weights are not row-centered)')
print('  - For row-centered: the forward gain is MULTIPLIED by this ratio')
print('  - Theoretical half-Gaussian value: (π-1)/π ≈', f'{(math.pi-1)/math.pi:.4f}')

## Section 3: ReLU Survival Analysis

Does row-centering change the fraction of neurons that survive ReLU?
Standard He predicts ~50% survival. Centering might change this.

In [ ]:
fig = plot_relu_survival(results_labeled)
plt.title('ReLU Survival Rate per Layer', fontsize=12)
plt.show()

# Summary statistics
for init in INIT_STRATEGIES:
    label = INIT_LABELS.get(init, init)
    rates = results_full[init].get_relu_survival_rates()
    vals = list(rates.values())
    print(f'{label}: survival = {np.mean(vals):.3f} ± {np.std(vals):.3f}')

## Section 4: Variance Sweep (Advisor's Coupling Observation)

From BP4: ∂C/∂W^l ∝ W¹ · ... · W^{l-1} · (W^{l+1})^T · ... · (W^L)^T

The advisor observed: changing variance at one layer affects gradients at other
layers differently depending on whether they're "forward" or "backward" from the
modified layer. We sweep variance factors to visualize this coupling.

For row-centered initialization, we expect:
- **Forward gain** scales linearly with variance (more variance → more signal)
- **Backward gain** ALSO scales with variance, but the RATIO forward/backward
  stays constant at Var(a)/E[a²] ≈ 0.68

This means no single variance can make both gains = 1 simultaneously.

In [ ]:
# Variance sweep for row-centered initialization
# We use custom_variance + manual row-centering via a helper
import torch.nn as nn
from rp_study.models.initializers import register_initializer, initialize_layer

VARIANCE_FACTORS = [1.5, 2.0, 2.5, 3.0]

variance_results = {}
for factor in VARIANCE_FACTORS:
    label = f'RC var={factor:.1f}/d'

    # Register a temporary initializer for this factor
    init_name = f'_sweep_rc_{factor}'

    @register_initializer(init_name)
    def _sweep_init(layer, _factor=factor, **kwargs):
        fan_in = layer.weight.shape[1]
        target_std = math.sqrt(_factor / fan_in)
        with torch.no_grad():
            layer.weight.normal_(0.0, target_std)
            layer.weight -= layer.weight.mean(dim=1, keepdim=True)
            row_stds = layer.weight.std(dim=1, keepdim=True, unbiased=False)
            row_stds = torch.clamp(row_stds, min=1e-8)
            layer.weight *= target_std / row_stds
            if layer.bias is not None:
                layer.bias.zero_()

    exp_config = ExperimentConfig(seed=SEED, data_dir=DATA_DIR)
    exp_config.setup_seeds()
    grad_config = GradientExperimentConfig(num_samples=NUM_SAMPLES_FULL, dataset=DATASET)
    net_config = NetworkConfig(layer_sizes=LAYER_SIZES, init_strategy=init_name)
    exp = GradientExperiment(exp_config, net_config, grad_config)
    variance_results[label] = exp.run()
    print(f'  {label}: done')

print('Variance sweep complete!')

In [ ]:
# Plot forward and backward gains for the variance sweep
fig = compare_gains_plot(variance_results, gain_type='forward')
plt.title('Forward Gain vs Variance Factor (row-centered)', fontsize=12)
plt.show()

fig = compare_gains_plot(variance_results, gain_type='backward')
plt.title('Backward Gain vs Variance Factor (row-centered)', fontsize=12)
plt.show()

# Summary: median forward and backward gains for each factor
print(f'{"Factor":>10s}  {"Median Fwd":>10s}  {"Median Bwd":>10s}  {"Ratio F/B":>10s}')
print('-' * 45)
for label, result in variance_results.items():
    fwd = list(result.get_forward_gains().values())
    bwd = list(result.get_backward_gains().values())
    # Exclude last hidden layer (boundary effect)
    fwd = fwd[:-1] if len(fwd) > 1 else fwd
    bwd = bwd[:-1] if len(bwd) > 1 else bwd
    med_fwd = np.median(fwd)
    med_bwd = np.median(bwd)
    ratio = med_fwd / med_bwd if med_bwd > 0 else float('nan')
    print(f'{label:>10s}  {med_fwd:>10.4f}  {med_bwd:>10.4f}  {ratio:>10.4f}')

print(f'\nTheoretical ratio (π-1)/π = {(math.pi-1)/math.pi:.4f}')
print('Note: The ratio is approximately constant — confirming that no single')
print('variance can simultaneously fix both forward and backward gains.')

## Section 5: Alpha Sweep for Partial Centering

Since the asymmetry is structural (caused by row-centering + ReLU), the only way to
reduce it is to use **partial centering**: `W = W_he - α · mean(W_he, dim=1)`.

- α = 0: standard He (perfect gradients, bad geometry)
- α = 1: full centering (perfect geometry, bad gradients)

We sweep α to find the Pareto-optimal trade-off.

In [ ]:
from sklearn.decomposition import PCA
from rp_study.data.loaders import get_data_loader
from rp_study.projections import multi_layer_rp_with_init

ALPHA_VALUES = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
GEOM_N_LAYERS = 20  # layers for geometry test

# Load data for geometry test
config = ExperimentConfig(seed=SEED, data_dir=DATA_DIR)
config.setup_seeds()
X_geom, y_geom = get_data_loader(
    dataset_name=DATASET, data_dir=DATA_DIR, train=True,
    flatten=True, as_numpy=True
)

alpha_results = {}   # gradient results
alpha_geometry = {}  # PCA projections for geometry

for alpha in ALPHA_VALUES:
    label = f'α={alpha:.1f}'
    init_name = f'_sweep_alpha_{alpha}'

    @register_initializer(init_name)
    def _alpha_init(layer, _alpha=alpha, **kwargs):
        fan_in = layer.weight.shape[1]
        target_std = math.sqrt(2.0 / fan_in)
        with torch.no_grad():
            layer.weight.normal_(0.0, target_std)
            layer.weight -= _alpha * layer.weight.mean(dim=1, keepdim=True)
            # Rescale to restore He variance
            row_stds = layer.weight.std(dim=1, keepdim=True, unbiased=False)
            row_stds = torch.clamp(row_stds, min=1e-8)
            layer.weight *= target_std / row_stds
            if layer.bias is not None:
                layer.bias.zero_()

    # Gradient analysis
    exp_config = ExperimentConfig(seed=SEED, data_dir=DATA_DIR)
    exp_config.setup_seeds()
    grad_config = GradientExperimentConfig(num_samples=NUM_SAMPLES_FULL, dataset=DATASET)
    net_config = NetworkConfig(layer_sizes=LAYER_SIZES, init_strategy=init_name)
    exp = GradientExperiment(exp_config, net_config, grad_config)
    alpha_results[label] = exp.run()

    # Geometry analysis (multi-layer RP + PCA)
    X_proj = multi_layer_rp_with_init(
        X_geom, GEOM_N_LAYERS, init_strategy=init_name,
        seed=SEED, device=DEVICE,
    )
    pca = PCA(n_components=2)
    alpha_geometry[label] = pca.fit_transform(X_proj)

    print(f'  {label}: done')

print('Alpha sweep complete!')

In [ ]:
# Geometry: PCA projections for each alpha
n_alphas = len(ALPHA_VALUES)
fig, axes = plt.subplots(1, n_alphas, figsize=(4 * n_alphas, 3.5))
if n_alphas == 1:
    axes = [axes]

for i, alpha in enumerate(ALPHA_VALUES):
    label = f'α={alpha:.1f}'
    X_vis = alpha_geometry[label]
    axes[i].scatter(X_vis[:, 0], X_vis[:, 1], s=2, c=y_geom, cmap='viridis', alpha=0.7)
    axes[i].set_title(label, fontsize=12)
    axes[i].set_xticks([])
    axes[i].set_yticks([])
    axes[i].axis('equal')

plt.suptitle(f'Geometry after {GEOM_N_LAYERS} Layers: Partial Centering Alpha Sweep',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Forward and backward gains for each alpha
fig = compare_gains_plot(alpha_results, gain_type='forward')
plt.title('Forward Gain vs Alpha', fontsize=12)
plt.show()

fig = compare_gains_plot(alpha_results, gain_type='backward')
plt.title('Backward Gain vs Alpha', fontsize=12)
plt.show()

In [ ]:
# Gradient norm profiles for each alpha
fig = compare_initializations_plot(
    alpha_results, metric='mean_row_norm',
    exclude_output_layer=True, use_log_scale=True
)
plt.title('Gradient Row Norms: Alpha Sweep (log scale)', fontsize=12)
plt.show()

In [ ]:
# Pareto frontier: gradient uniformity vs geometry quality
print(f'{"Alpha":>6s}  {"Med Fwd":>8s}  {"Med Bwd":>8s}  {"Grad Spread":>12s}  {"Geometry":>10s}')
print('-' * 55)

for alpha in ALPHA_VALUES:
    label = f'α={alpha:.1f}'
    result = alpha_results[label]

    # Gains (exclude last hidden layer)
    fwd = list(result.get_forward_gains().values())[:-1]
    bwd = list(result.get_backward_gains().values())[:-1]
    med_fwd = np.median(fwd)
    med_bwd = np.median(bwd)

    # Gradient spread: ratio of max to min row norm (hidden layers)
    norms = list(result.get_mean_row_norms().values())
    norms = norms[:-1]  # exclude output
    if min(norms) > 0:
        spread = max(norms) / min(norms)
    else:
        spread = float('inf')

    # Geometry: check if PCA projection has meaningful structure
    X_vis = alpha_geometry[label]
    # Simple metric: variance explained by first PC relative to total
    geom_quality = np.var(X_vis[:, 0]) / (np.var(X_vis[:, 0]) + np.var(X_vis[:, 1]) + 1e-10)

    print(f'{alpha:>6.1f}  {med_fwd:>8.4f}  {med_bwd:>8.4f}  {spread:>12.1f}x  {geom_quality:>10.3f}')

print()
print('Goal: Find alpha with low gradient spread AND good geometry.')
print('Gradient spread close to 1x = uniform gradients across layers.')

## Summary

Key findings from this diagnostic notebook:

1. **Single data point**: [Run the notebook to see if the pattern holds]
2. **Forward-backward asymmetry**: Row-centering makes forward gain < backward gain
   because the weights operate on centered activations (removing mean E[a] > 0)
3. **Centering ratio**: Var(a)/E[a²] ≈ 0.68 for post-ReLU, confirming the theoretical
   prediction of (π-1)/π
4. **Variance coupling**: The forward/backward gain ratio is constant ≈ 0.68 regardless
   of variance — confirming no scalar fix exists
5. **Partial centering**: Alpha sweep reveals the Pareto frontier between geometry
   preservation and gradient stability